In [ ]:
import os
import yaml
import torch
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
from deeplearning.models import AttentionDiffusionRNN
from deeplearning.dataloader import WordDataLoader
from IPython.display import clear_output

class AttentionModelTrainer:
    def __init__(self, config_path='./deeplearning/config.yaml'):
        """Initialize the trainer with configuration"""
        self.setup_config(config_path)
        self.setup_cuda()
        self.setup_paths()
        self.setup_data()
        self.setup_model()
        self.setup_metrics()
        
    def setup_config(self, config_path):
        """Load and validate configuration"""
        with open(config_path, 'r') as stream:
            self.config = yaml.safe_load(stream)
        
        # Ensure all required configs are present
        required_configs = ['cuda', 'epochs', 'batch_size', 'num_workers']
        for config in required_configs:
            if config not in self.config:
                raise ValueError(f"Missing required config: {config}")
    
    def setup_cuda(self):
        """Setup CUDA if available"""
        self.cuda = torch.cuda.is_available() and self.config['cuda']
        if self.cuda:
            device_id = 3  # Using GPU 3 as in original code
            torch.cuda.set_device(device_id)
            print(f"Using CUDA device {device_id}: {torch.cuda.get_device_name()}")
        else:
            print("Using CPU")
    
    def setup_paths(self):
        """Setup directory structure for model artifacts"""
        self.model_name = f"{self.config['rnn']}_{self.config['num_layers']}_{self.config['hidden_dim']}"
        
        for path_type in ['models', 'plots']:
            full_path = os.path.join(self.config[path_type], self.model_name)
            os.makedirs(full_path, exist_ok=True)
            setattr(self, f"{path_type}_path", full_path)
    
    def setup_data(self):
        """Initialize data loaders"""
        try:
            self.train_loader = WordDataLoader('train', self.config)
            self.val_loader = WordDataLoader('test', self.config)
        except Exception as e:
            raise RuntimeError(f"Failed to initialize data loaders: {str(e)}")
    
    def setup_model(self):
        """Initialize the model"""
        try:
            self.model = AttentionDiffusionRNN(self.config)
            if self.cuda:
                self.model = self.model.cuda()
        except Exception as e:
            raise RuntimeError(f"Failed to initialize model: {str(e)}")
    
    def setup_metrics(self):
        """Initialize training metrics"""
        self.train_losses = []
        self.val_losses = []
        self.attention_metrics = []
        self.best_val_loss = float('inf')
    
    def process_batch(self, inputs, labels, miss_chars, input_lens):
        """Process a single batch of data"""
        # Convert numpy arrays to torch tensors
        if self.config['use_embedding']:
            inputs = torch.from_numpy(inputs).long()
        else:
            inputs = torch.from_numpy(inputs).float()
            
        labels = torch.from_numpy(labels).float()
        miss_chars = torch.from_numpy(miss_chars).float()
        input_lens = torch.from_numpy(input_lens).long()
        
        # Move to GPU if available
        if self.cuda:
            inputs = inputs.cuda()
            labels = labels.cuda()
            miss_chars = miss_chars.cuda()
            input_lens = input_lens.cuda()
            
        return inputs, labels, miss_chars, input_lens
    
    def train_epoch(self, epoch):
        """Train for one epoch"""
        self.model.train()
        epoch_loss = 0.0
        attention_loss = 0.0
        
        pbar = tqdm(self.train_loader, desc=f'Epoch [{epoch}/{self.config["epochs"]}]')
        
        for batch_idx, (inputs, labels, miss_chars, input_lens) in enumerate(pbar):
            try:
                # Process batch
                inputs, labels, miss_chars, input_lens = self.process_batch(
                    inputs, labels, miss_chars, input_lens
                )
                
                # Forward pass
                self.model.optimizer.zero_grad()
                outputs = self.model(inputs, input_lens, miss_chars)
                loss, miss_penalty = self.model.calculate_loss(
                    outputs, labels, input_lens, miss_chars, self.cuda
                )
                
                # Backward pass
                loss.backward()
                self.model.optimizer.step()
                
                # Update metrics
                epoch_loss += loss.item()
                attention_loss += miss_penalty.item() if isinstance(miss_penalty, torch.Tensor) else miss_penalty
                
                # Update progress bar
                avg_loss = epoch_loss / (batch_idx + 1)
                avg_att_loss = attention_loss / (batch_idx + 1)
                pbar.set_postfix({
                    'loss': f'{avg_loss:.4f}',
                    'att_loss': f'{avg_att_loss:.4f}'
                })
                
            except Exception as e:
                print(f"Error processing batch {batch_idx}: {str(e)}")
                continue
        
        return epoch_loss / len(self.train_loader), attention_loss / len(self.train_loader)
    
    def validate(self):
        """Validate the model"""
        self.model.eval()
        val_loss = 0.0
        attention_loss = 0.0
        
        with torch.no_grad():
            for inputs, labels, miss_chars, input_lens in tqdm(self.val_loader, desc='Validating'):
                try:
                    # Process batch
                    inputs, labels, miss_chars, input_lens = self.process_batch(
                        inputs, labels, miss_chars, input_lens
                    )
                    
                    # Forward pass
                    outputs = self.model(inputs, input_lens, miss_chars)
                    loss, miss_penalty = self.model.calculate_loss(
                        outputs, labels, input_lens, miss_chars, self.cuda
                    )
                    
                    val_loss += loss.item()
                    attention_loss += miss_penalty.item() if isinstance(miss_penalty, torch.Tensor) else miss_penalty
                    
                except Exception as e:
                    print(f"Error during validation: {str(e)}")
                    continue
        
        val_loss = val_loss / len(self.val_loader)
        attention_loss = attention_loss / len(self.val_loader)
        
        print(f'\nValidation - Loss: {val_loss:.4f}, Attention Loss: {attention_loss:.4f}')
        return val_loss, attention_loss
    
    def save_checkpoint(self, epoch, train_loss, val_loss=None, is_best=False):
        """Save model checkpoint"""
        try:
            self.model.save_model(
                is_best=is_best,
                epoch=epoch,
                train_loss=train_loss,
                test_loss=val_loss if val_loss is not None else train_loss,
                rnn_name=self.config['rnn'],
                layers=self.config['num_layers'],
                hidden_dim=self.config['hidden_dim']
            )
        except Exception as e:
            print(f"Error saving checkpoint: {str(e)}")
    
    def plot_metrics(self, save=True):
        """Plot training metrics"""
        try:
            plt.figure(figsize=(12, 4))
            
            # Plot losses
            plt.subplot(1, 2, 1)
            plt.plot(self.train_losses, label='Train Loss')
            plt.plot(self.val_losses, label='Validation Loss')
            plt.title('Training and Validation Loss')
            plt.xlabel('Epoch')
            plt.ylabel('Loss')
            plt.legend()
            
            # Plot attention metrics
            plt.subplot(1, 2, 2)
            plt.plot(self.attention_metrics, label='Attention Loss')
            plt.title('Attention Loss')
            plt.xlabel('Epoch')
            plt.ylabel('Loss')
            plt.legend()
            
            if save:
                plt.savefig(os.path.join(self.plots_path, f'training_plot.png'))
            plt.show()
            
        except Exception as e:
            print(f"Error plotting metrics: {str(e)}")
    
    def train(self):
        """Main training loop"""
        print(f"Starting training for {self.config['epochs']} epochs...")
        
        try:
            for epoch in range(1, self.config['epochs'] + 1):
                # Training
                train_loss, train_att_loss = self.train_epoch(epoch)
                self.train_losses.append(train_loss)
                
                # Validation
                if epoch % self.config['test_every_epoch'] == 0:
                    val_loss, val_att_loss = self.validate()
                    self.val_losses.append(val_loss)
                    self.attention_metrics.append(val_att_loss)
                    
                    # Save best model
                    if val_loss < self.best_val_loss:
                        self.best_val_loss = val_loss
                        self.save_checkpoint(epoch, train_loss, val_loss, is_best=True)
                    
                    # Plot metrics
                    if epoch % self.config['plot_every'] == 0:
                        clear_output(wait=True)
                        self.plot_metrics()
                
                # Save periodic checkpoint
                if epoch % self.config['save_every'] == 0:
                    self.save_checkpoint(
                        epoch, 
                        train_loss,
                        val_loss if len(self.val_losses) > 0 else None
                    )
                
                # Update dataset
                self.train_loader.update_dataset(epoch)
                
        except KeyboardInterrupt:
            print("\nTraining interrupted by user. Saving checkpoint...")
            self.save_checkpoint(epoch, train_loss, val_loss if len(self.val_losses) > 0 else None)
            
        except Exception as e:
            print(f"Error during training: {str(e)}")
            raise
        
        print("Training completed!")

# Example usage
if __name__ == "__main__":
    try:
        # Initialize trainer
        trainer = AttentionModelTrainer()
        
        # Start training
        trainer.train()
        
    except Exception as e:
        print(f"Fatal error: {str(e)}")

Length of train dataset: 227019


/Users/famadeo/anaconda3/lib/python3.11/site-packages/torch/utils/data/dataloader.py:558: UserWarning: This DataLoader will create 64 worker processes in total. Our suggested max number of worker in current system is 16 (`cpuset` is not taken into account), which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(_create_warning_msg(


Length of test dataset: 184906


2025-01-15 11:58:16.539238: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Epoch [1/200]:   0%|          | 0/111 [00:00<?, ?it/s]/Users/famadeo/anaconda3/lib/python3.11/site-packages/torch/utils/data/dataloader.py:558: UserWarning: This DataLoader will create 64 worker processes in total. Our suggested max number of worker in current system is 16 (`cpuset` is not taken into account), which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(_create_warning_msg(
Epoch [1/200]:   0%|          | 0/111 [00:02<?, ?it/s]


AttributeError: Can't pickle local object 'WordDataLoader.__init__.<locals>.<lambda>'